
## Imports

In [ ]:
import subprocess, sys
for pkg in ['xgboost', 'imbalanced-learn', 'nltk', 'wordcloud']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print("Libraries ready")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# NLP
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

# ML
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve,
                              precision_score, recall_score, f1_score)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib
from scipy.sparse import hstack, csr_matrix

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
print("All imports successful")


## Dataset Load


In [ ]:
train_df = pd.read_csv(r"C:\Users\Share\Documents\Final Year Degree (Software Engineer)\Second Semester\Computational Intelligence\Fake Job Posting Detection Dataset for the Competition\job_postings_train.csv")
test_df  = pd.read_csv(r"C:\Users\Share\Documents\Final Year Degree (Software Engineer)\Second Semester\Computational Intelligence\Fake Job Posting Detection Dataset for the Competition\job_postings_test.csv")

print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")
print(f"\nColumns:\n{list(train_df.columns)}")
train_df.head(5)


## Exploratory Data Analysis (EDA)


In [ ]:
#1. Identifing data types
print("=== Data Types ===")
print(train_df.dtypes)

In [ ]:
#2. Target class distribution
print(f"\n=== Target Distribution ===")
vc = train_df['fraudulent'].value_counts()
print(f"Real (0)  : {vc[0]} ({vc[0]/len(train_df)*100:.1f}%)")
print(f"Fake (1)  : {vc[1]} ({vc[1]/len(train_df)*100:.1f}%)")
print(f"\n Class imbalance ratio: {vc[0]/vc[1]:.1f}:1  (Real:Fake)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
vc = train_df['fraudulent'].value_counts()

axes[0].bar(['Real (0)', 'Fake (1)'], vc.values, color=['#2ecc71', '#e74c3c'],
            edgecolor='white', linewidth=1.5)
axes[0].set_title('Target Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 50, f'{v}\n({v/len(train_df)*100:.1f}%)', ha='center', fontweight='bold')

axes[1].pie(vc.values, labels=['Real Jobs', 'Fake Jobs'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Fraud Rate Distribution', fontweight='bold')

plt.suptitle('EDA 2 - Target Variable: fraudulent', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_class_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
#3. Missing values analysis
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(1)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0].sort_values('Percentage', ascending=False)

print("=== Missing Values ===")
print(missing_df)

plt.figure(figsize=(10, 5))
missing_df['Percentage'].plot(kind='barh', color='#e74c3c', alpha=0.8)
plt.title('EDA 3 - Missing Values by Column (%)', fontweight='bold', fontsize=13)
plt.xlabel('Missing %')
plt.tight_layout()
plt.savefig('eda_missing_values.png', bbox_inches='tight')
plt.show()


In [ ]:
#4. Binary feature analysis vs fraud
binary_cols = ['telecommuting', 'has_company_logo', 'has_questions']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col in zip(axes, binary_cols):
    fraud_rate = train_df.groupby(col)['fraudulent'].mean() * 100
    bars = ax.bar(fraud_rate.index.astype(str), fraud_rate.values,
                  color=['#3498db', '#e74c3c'], alpha=0.85, edgecolor='white')
    ax.set_title(f'Fraud Rate by\n{col}', fontweight='bold')
    ax.set_ylabel('Fraud Rate (%)')
    ax.set_xlabel(col)
    for bar, val in zip(bars, fraud_rate.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                f'{val:.1f}%', ha='center', fontweight='bold')

plt.suptitle('EDA 4 - Binary Features vs Fraud Rate', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_binary_features.png', bbox_inches='tight')
plt.show()


In [ ]:
#5. Text length analysis (description length as fraud indicator)
train_df['desc_len']  = train_df['description'].fillna('').apply(len)
train_df['title_len'] = train_df['title'].fillna('').apply(len)
train_df['req_len']   = train_df['requirements'].fillna('').apply(len)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, label in zip(axes,
    ['desc_len', 'title_len', 'req_len'],
    ['Description Length', 'Title Length', 'Requirements Length']):
    for fraud_val, color, lbl in [(0,'#2ecc71','Real'), (1,'#e74c3c','Fake')]:
        data = train_df[train_df['fraudulent']==fraud_val][col]
        ax.hist(data.clip(upper=data.quantile(0.95)), bins=40,
                alpha=0.6, color=color, label=lbl, density=True)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Characters')
    ax.legend()

plt.suptitle('EDA 5 - Text Length: Real vs Fake Jobs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_text_length.png', bbox_inches='tight')
plt.show()

In [ ]:
# 6. Top industries with highest fraud rates
industry_fraud = train_df.groupby('industry').agg(
    total=('fraudulent','count'),
    fraud_count=('fraudulent','sum')
).reset_index()
industry_fraud['fraud_rate'] = industry_fraud['fraud_count'] / industry_fraud['total'] * 100
industry_fraud = industry_fraud[industry_fraud['total'] >= 50].sort_values('fraud_rate', ascending=False).head(10)

plt.figure(figsize=(10, 5))
plt.barh(industry_fraud['industry'], industry_fraud['fraud_rate'],
         color='#e74c3c', alpha=0.8)
plt.title('EDA 6 - Top 10 Industries by Fraud Rate', fontweight='bold', fontsize=12)
plt.xlabel('Fraud Rate (%)')
plt.tight_layout()
plt.savefig('eda_industry_fraud.png', bbox_inches='tight')
plt.show()

In [ ]:
#7. Word frequency (top words in real vs fake job postings)
from collections import Counter
import re as re2

stop = set(stopwords.words('english'))

def get_top_words(df, label, n=15):
    texts = df[df['fraudulent'] == label]['description'].fillna('').str.lower()
    words = re2.sub(r'[^a-z\s]', ' ', ' '.join(texts)).split()
    words = [w for w in words if w not in stop and len(w) > 3]
    return Counter(words).most_common(n)

real_words = get_top_words(train_df, 0)
fake_words = get_top_words(train_df, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

words_r, counts_r = zip(*real_words)
words_f, counts_f = zip(*fake_words)

axes[0].barh(list(words_r)[::-1], list(counts_r)[::-1], color='#2ecc71', alpha=0.85, edgecolor='white')
axes[0].set_title('Top 15 Words - REAL Job Postings', fontweight='bold')
axes[0].set_xlabel('Frequency')

axes[1].barh(list(words_f)[::-1], list(counts_f)[::-1], color='#e74c3c', alpha=0.85, edgecolor='white')
axes[1].set_title('Top 15 Words - FAKE Job Postings', fontweight='bold')
axes[1].set_xlabel('Frequency')

plt.suptitle('EDA 7 - Word Frequency: Real vs Fake Job Descriptions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_word_frequency.png', bbox_inches='tight')
plt.show()


In [ ]:
#8. Employment type and required experience vs fraud rate
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

cat_cols = ['employment_type', 'required_experience']
cat_labels = ['Employment Type', 'Required Experience']

for ax, col, label in zip(axes, cat_cols, cat_labels):
    temp = train_df.copy()
    temp[col] = temp[col].fillna('Unknown')
    grp = temp.groupby(col).agg(
        total=('fraudulent', 'count'),
        fraud=('fraudulent', 'sum')
    ).reset_index()
    grp['fraud_rate'] = grp['fraud'] / grp['total'] * 100
    grp = grp[grp['total'] >= 30].sort_values('fraud_rate', ascending=True)

    bars = ax.barh(grp[col], grp['fraud_rate'], color='#e74c3c', alpha=0.8, edgecolor='white')
    for bar, (_, row) in zip(bars, grp.iterrows()):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f"{row['fraud_rate']:.1f}% (n={row['total']})", va='center', fontsize=8)
    ax.set_title(f'Fraud Rate by {label}', fontweight='bold')
    ax.set_xlabel('Fraud Rate (%)')

plt.suptitle('EDA 8 - Employment Type & Experience vs Fraud Rate', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_employment_fraud.png', bbox_inches='tight')
plt.show()



## Preprocessing

In [ ]:
#Define text cleaning function
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if pd.isna(text) or text == '':
        return ''
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)          # Remove HTML tags
    text = re.sub(r'http\S+|www\S+', ' ', text)  # Remove URLs
    text = re.sub(r'[^a-z\s]', ' ', text)         # Keep only letters
    text = re.sub(r'\s+', ' ', text).strip()       # Remove extra spaces
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in STOP_WORDS and len(w) > 2]
    return ' '.join(tokens)

# Test
print("Sample:", clean_text("We are hiring a <b>Senior Engineer</b> for $100K! Apply now at www.jobs.com"))

In [ ]:
#Preprocess train and test sets
def preprocess(df, is_train=True):
    df = df.copy()

    # Drop salary_range (83.6% missing)
    if 'salary_range' in df.columns:
        df.drop('salary_range', axis=1, inplace=True)

    # Fill missing text columns with empty string
    text_cols = ['title', 'location', 'department', 'company_profile',
                 'description', 'requirements', 'benefits',
                 'employment_type', 'required_experience',
                 'required_education', 'industry', 'function']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna('')

    # Fill binary columns with 0
    for col in ['telecommuting', 'has_company_logo', 'has_questions']:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)

    # Engineer: text length features
    df['desc_len']  = df['description'].apply(len)
    df['title_len'] = df['title'].apply(len)
    df['req_len']   = df['requirements'].apply(len)
    df['has_desc']  = (df['description'] != '').astype(int)
    df['has_req']   = (df['requirements'] != '').astype(int)
    df['has_comp']  = (df['company_profile'] != '').astype(int)

    # Combine all text for TF-IDF
    df['full_text'] = (df['title'] + ' ' + df['company_profile'] + ' ' +
                       df['description'] + ' ' + df['requirements'] + ' ' +
                       df['benefits'] + ' ' + df['employment_type'] + ' ' +
                       df['required_experience'] + ' ' + df['industry'])

    df['full_text_clean'] = df['full_text'].apply(clean_text)

    return df


train_proc = preprocess(train_df, is_train=True)
test_proc  = preprocess(test_df,  is_train=False)

print(f"\n Train processed: {train_proc.shape}")
print(f" Test processed : {test_proc.shape}")
print(f"\nNew features added: desc_len, title_len, req_len, has_desc, has_req, has_comp, full_text_clean")

In [ ]:
#TF-IDF Vectorisation of combined text
print("Fitting TF-IDF vectoriser...")
tfidf = TfidfVectorizer(
    max_features=5000,       # Top 5000 most informative words
    ngram_range=(1, 2),      # Unigrams + bigrams
    min_df=3,                # Ignore terms in fewer than 3 docs
    sublinear_tf=True        # Apply log normalisation
)
X_tfidf_train = tfidf.fit_transform(train_proc['full_text_clean'])
X_tfidf_test  = tfidf.transform(test_proc['full_text_clean'])

print(f"TF-IDF matrix shape: {X_tfidf_train.shape}")
print(f"Top 20 most important terms:")
feature_names = tfidf.get_feature_names_out()
print(list(feature_names[:20]))

In [ ]:
#Combine TF-IDF + numeric/binary features
NUMERIC_COLS = ['telecommuting', 'has_company_logo', 'has_questions',
                'desc_len', 'title_len', 'req_len',
                'has_desc', 'has_req', 'has_comp']

X_num_train = csr_matrix(train_proc[NUMERIC_COLS].values.astype(float))
X_num_test  = csr_matrix(test_proc[NUMERIC_COLS].values.astype(float))

# Stack TF-IDF + numeric features horizontally
X_train_full = hstack([X_tfidf_train, X_num_train])
X_test_full  = hstack([X_tfidf_test,  X_num_test])

y = train_proc['fraudulent']
test_ids = test_proc['id']

print(f"Combined feature matrix shape: {X_train_full.shape}")
print(f"Total features: {X_train_full.shape[1]} (5000 TF-IDF + {len(NUMERIC_COLS)} numeric)")

In [ ]:
#Train/validation split + SMOTE
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_tr.shape[0]} | Val: {X_val.shape[0]}")
print(f"Train fraud rate: {y_tr.mean()*100:.1f}%")

# Apply SMOTE on training set
smote = SMOTE(random_state=42)
X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)

print(f"\nBefore SMOTE - Real: {sum(y_tr==0)} | Fake: {sum(y_tr==1)}")
print(f"After  SMOTE - Real: {sum(y_tr_sm==0)} | Fake: {sum(y_tr_sm==1)}")


## Model Training & Full Evaluation

In [ ]:
#models
models = {
    "Logistic Regression": LogisticRegression(
        C=1.0, max_iter=1000, random_state=42, n_jobs=-1),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric='logloss', verbosity=0),
     "SVM (Linear)": CalibratedClassifierCV(
        LinearSVC(C=1.0, max_iter=2000, random_state=42),
        cv=3
    ),
   "Neural Network": MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    tol=1e-4
),
}
print("Models defined")
for name in models: print(f"   {name}")

In [ ]:
#Train and evaluate all models
results = {}
print("Training models...\n")

for name, model in models.items():
    print(f" {name}...", end=" ")
    model.fit(X_tr_sm, y_tr_sm)
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]

    results[name] = {
        'model':    model,
        'accuracy': accuracy_score(y_val, y_pred),
        'auc':      roc_auc_score(y_val, y_prob),
        'precision':precision_score(y_val, y_pred, zero_division=0),
        'recall':   recall_score(y_val, y_pred, zero_division=0),
        'f1':       f1_score(y_val, y_pred, zero_division=0),
        'y_pred':   y_pred,
        'y_prob':   y_prob,
    }
    print(f"F1={results[name]['f1']:.4f} | AUC={results[name]['auc']:.4f}")

print("\nAll models trained")

In [ ]:
#Model comparison charts
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

names   = list(results.keys())
metrics = ['accuracy', 'auc', 'f1']
colors  = ['#3498db', '#e74c3c', '#2ecc71']
labels  = ['Accuracy', 'AUC-ROC', 'F1 Score']
x, w = np.arange(len(names)), 0.25

for i, (m, c, l) in enumerate(zip(metrics, colors, labels)):
    vals = [results[n][m] for n in names]
    axes[0].bar(x+i*w, vals, w, label=l, color=c, alpha=0.85)
axes[0].set_xticks(x+w)
axes[0].set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=8)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Metrics Comparison')
axes[0].legend(fontsize=9)
axes[0].set_ylabel('Score')

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_val, res['y_prob'])
    axes[1].plot(fpr, tpr, linewidth=2, label=f"{name} ({res['auc']:.3f})")
axes[1].plot([0,1],[0,1],'k--', alpha=0.4)
axes[1].set_title('ROC Curves')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight', dpi=130)
plt.show()

## Hyperparameter Tuning

In [ ]:
#hyperparameter tuning
#GridSearchCV - Logistic Regression

param_grid_lr = {
    'C':        [0.01, 0.1, 1.0, 10.0],
    'max_iter': [500, 1000]
}

grid_lr = GridSearchCV(
    LogisticRegression(random_state=42, n_jobs=-1),
    param_grid_lr,
    cv=5, scoring='f1', n_jobs=-1, verbose=0
)
grid_lr.fit(X_tr_sm, y_tr_sm)

print(f"   Best Params : {grid_lr.best_params_}")
print(f"   Best CV F1  : {grid_lr.best_score_:.4f}")
best_lr_tuned = grid_lr.best_estimator_

In [ ]:
#GridSearchCV - SVM

param_grid_svm = {
     'estimator__C':        [0.01, 0.1, 1.0, 10.0],
     'estimator__max_iter': [1000, 2000]
}

grid_svm = GridSearchCV(
    CalibratedClassifierCV(LinearSVC(random_state=42), cv=3),
    param_grid_svm,
    cv=5, scoring='f1', n_jobs=-1, verbose=0
)
grid_svm.fit(X_tr_sm, y_tr_sm)

print(f"   Best Params : {grid_svm.best_params_}")
print(f"   Best CV F1  : {grid_svm.best_score_:.4f}")
best_svm_tuned = grid_svm.best_estimator_

In [ ]:
#RandomizedSearchCV - Random Forest

param_dist_rf = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth':    [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rand_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist_rf,
    n_iter=20, cv=5, scoring='f1',
    n_jobs=-1, random_state=42, verbose=0
)
rand_rf.fit(X_tr_sm, y_tr_sm)

print(f"   Best Params : {rand_rf.best_params_}")
print(f"   Best CV F1  : {rand_rf.best_score_:.4f}")
best_rf_tuned = rand_rf.best_estimator_

In [ ]:
#Compare Tuned vs Original

print(f"  TUNED MODEL COMPARISON")
print(f"{'Model':<30} {'F1':>8} {'AUC':>8} {'Precision':>10} {'Recall':>8}")


tuned_models = {
    "LR  (GridSearch Tuned)":  best_lr_tuned,
    "SVM (GridSearch Tuned)":  best_svm_tuned,
    "RF  (Random Tuned)":      best_rf_tuned,
}

tuned_results = {}
for name, model in tuned_models.items():
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]
    tuned_results[name] = {
        'accuracy':  accuracy_score(y_val, y_pred),
        'f1':        f1_score(y_val, y_pred, zero_division=0),
        'auc':       roc_auc_score(y_val, y_prob),
        'precision': precision_score(y_val, y_pred, zero_division=0),
        'recall':    recall_score(y_val, y_pred, zero_division=0),
        'model':     model,
        'y_pred':    y_pred,
        'y_prob':    y_prob,
    }
    r = tuned_results[name]
    print(f"{name:<30} {r['f1']:>8.4f} {r['auc']:>8.4f} {r['precision']:>10.4f} {r['recall']:>8.4f}")

best_tuned_name  = max(tuned_results, key=lambda x: tuned_results[x]['f1'])
best_tuned_model = tuned_results[best_tuned_name]['model']
print(f"\n  Best Tuned Model: {best_tuned_name} → F1={tuned_results[best_tuned_name]['f1']:.4f}")

In [ ]:
#tuned Model comparison charts
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Tuned Model Performance Comparison', fontsize=14, fontweight='bold')

names   = list(tuned_results.keys())
metrics = ['accuracy', 'auc', 'f1']
colors  = ['#3498db', '#e74c3c', '#2ecc71']
labels  = ['Accuracy', 'AUC-ROC', 'F1 Score']
x, w = np.arange(len(names)), 0.25

for i, (m, c, l) in enumerate(zip(metrics, colors, labels)):
    vals = [tuned_results[n][m] for n in names]
    axes[0].bar(x+i*w, vals, w, label=l, color=c, alpha=0.85)
axes[0].set_xticks(x+w)
axes[0].set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=8)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Metrics Comparison')
axes[0].legend(fontsize=9)
axes[0].set_ylabel('Score')

for name, res in tuned_results.items():
    fpr, tpr, _ = roc_curve(y_val, res['y_prob'])
    axes[1].plot(fpr, tpr, linewidth=2, label=f"{name} ({res['auc']:.3f})")
axes[1].plot([0,1],[0,1],'k--', alpha=0.4)
axes[1].set_title('ROC Curves')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('tuned_model_comparison.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
#Comparison table (Original + Tuned Models)
print(f"\n{'='*80}")
print(f"  ORIGINAL MODELS (Before Tuning)")
print(f"{'='*80}")
print(f"{'Model':<25} {'Accuracy':>10} {'AUC-ROC':>10} {'Precision':>10} {'Recall':>10} {'F1':>8}")
print(f"{'─'*80}")
for name, r in results.items():
    print(f"{name:<25} {r['accuracy']:>10.4f} {r['auc']:>10.4f} {r['precision']:>10.4f} {r['recall']:>10.4f} {r['f1']:>8.4f}")

print(f"\n{'='*80}")
print(f"  TUNED MODELS (After Hyperparameter Tuning)")
print(f"{'='*80}")
print(f"{'Model':<25} {'Accuracy':>10} {'AUC-ROC':>10} {'Precision':>10} {'Recall':>10} {'F1':>8}")
print(f"{'─'*80}")
for name, r in tuned_results.items():
    print(f"{name:<25} {r['accuracy']:>10.4f} {r['auc']:>10.4f} {r['precision']:>10.4f} {r['recall']:>10.4f} {r['f1']:>8.4f}")

print(f"\n{'='*80}")
print(f"  BEFORE vs AFTER TUNING - F1 IMPROVEMENT")
print(f"{'='*80}")
print(f"{'Model':<25} {'Before F1':>12} {'After F1':>12} {'Change':>10}")
print(f"{'─'*80}")

# Map original names to tuned names for comparison
name_map = {
    "LR  (GridSearch Tuned)":  "Logistic Regression",
    "SVM (GridSearch Tuned)":  "SVM (Linear)",
    "RF  (Random Tuned)":      "Random Forest",
}

for tuned_name, orig_name in name_map.items():
    before = results[orig_name]['f1']
    after  = tuned_results[tuned_name]['f1']
    change = after - before
    print(f"{orig_name:<25} {before:>12.4f} {after:>12.4f} {change:>+12.4f} ")

# Best overall - compare across both original and tuned
print(f"\n{'='*80}")
best_original_name  = max(results, key=lambda x: results[x]['f1'])
best_original_f1    = results[best_original_name]['f1']
best_tuned_name     = max(tuned_results, key=lambda x: tuned_results[x]['f1'])
best_tuned_f1       = tuned_results[best_tuned_name]['f1']

print(f" Best Original : {best_original_name:<25} → F1 = {best_original_f1:.4f}")
print(f" Best Tuned    : {best_tuned_name:<25} → F1 = {best_tuned_f1:.4f}")

# Set best model for downstream use
if best_tuned_f1 >= best_original_f1:
    best_name  = best_tuned_name
    best_model = tuned_results[best_tuned_name]['model']
    print(f"\n Using TUNED model as final best: {best_name}")
else:
    best_name  = best_original_name
    best_model = results[best_original_name]['model']
    print(f"\n Using ORIGINAL model as final best: {best_name}")

In [ ]:
#Detailed classification report for best model
print(f"\n=== Classification Report: {best_name} ===")
print(classification_report(y_val, tuned_results[best_name]['y_pred'],
      target_names=['Real (0)', 'Fake (1)']))

In [ ]:
#Confusion matrices for all tuned models
fig, axes = plt.subplots(1, 3, figsize=(22, 4))
for ax, (name, res) in zip(axes, tuned_results.items()):
    cm = confusion_matrix(y_val, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Real','Fake'], yticklabels=['Real','Fake'],
                linewidths=1, linecolor='white')
    ax.set_title(f"{name}\nF1={res['f1']:.3f}", fontsize=9, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices - All Tuned Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
#Cross-validation (5-fold) for best model
print(f"=== 5-Fold Stratified Cross-Validation: {best_name} ===")
cv_scores = cross_val_score(
    best_model, X_train_full, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1', n_jobs=-1)

print(f"CV F1 Scores : {cv_scores.round(4)}")
print(f"Mean F1      : {cv_scores.mean():.4f}")
print(f"Std F1       : {cv_scores.std():.4f}")
print("\nStable" if cv_scores.std() < 0.05 else "High variance - consider regularisation")


## Kaggle Submissions



In [ ]:
def make_submission(model, X_test, test_ids, filename, label):
    preds = model.predict(X_test)
    sub = pd.DataFrame({'job_id': test_ids, 'fraudulent': preds})
    sub.to_csv(filename, index=False)
    print(f" {filename} — Fake: {sum(preds==1)} | Real: {sum(preds==0)}")
    print(f"   Upload to Kaggle as: '{label}'")
    return sub

In [ ]:
#Submission 1 - Logistic Regression (Baseline)
print("\n--- Submission 1: Logistic Regression Baseline ---")
lr_base = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_base.fit(X_tr, y_tr)  # No SMOTE for baseline
s1 = make_submission(lr_base, X_test_full, test_ids,
    'Submissions/submission_1.csv', 'Sub 1 - Logistic Regression Baseline')

In [ ]:
#Submission 2 - Logistic Regression + SMOTE
print("\n--- Submission 2: Logistic Regression + SMOTE ---")
lr_smote = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_smote.fit(X_tr_sm, y_tr_sm)
s2 = make_submission(lr_smote, X_test_full, test_ids,
    'Submissions/submission_2.csv', 'Sub 2 - Logistic Regression + SMOTE')

In [ ]:
#Submission 3 - Random Forest + SMOTE
print("\n--- Submission 3: Random Forest + SMOTE ---")
rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_tr_sm, y_tr_sm)
s3 = make_submission(rf, X_test_full, test_ids,
    'Submissions/submission_3.csv', 'Sub 3 - Random Forest + SMOTE')

In [ ]:
#Submission 4 - XGBoost + SMOTE
print("\n--- Submission 4: XGBoost + SMOTE ---")
xgb = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                     subsample=0.8, colsample_bytree=0.8,
                     random_state=42, eval_metric='logloss', verbosity=0)
xgb.fit(X_tr_sm, y_tr_sm)
s4 = make_submission(xgb, X_test_full, test_ids,
    'Submissions/submission_4.csv', 'Sub 4 - XGBoost + SMOTE')

In [ ]:
#Submission 5 - SVM Linear + SMOTE
print("\n--- Submission 5: SVM Linear + SMOTE ---")
svm = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm.fit(X_tr_sm, y_tr_sm)
s5 = make_submission(svm, X_test_full, test_ids,
    'Submissions/submission_5.csv', 'Sub 5 - SVM Linear + SMOTE')

In [ ]:
#Submission 6 - Neural Network + SMOTE
print("\n--- Submission 6: Neural Network MLP + SMOTE ---")
mlp = MLPClassifier(hidden_layer_sizes=(256, 128, 64), activation='relu',
                     solver='adam', learning_rate='adaptive',
                     max_iter=300, random_state=42)
mlp.fit(X_tr_sm, y_tr_sm)
s6 = make_submission(mlp, X_test_full, test_ids,
    'Submissions/submission_6.csv', 'Sub 6 - Neural Network MLP + SMOTE')


In [ ]:
#Submission 7 - Logistic Regression GridSearch Tuned
print("\n--- Submission 7: LR GridSearch Tuned ---")

s7 = make_submission(
    best_lr_tuned,
    X_test_full,
    test_ids,
    'Submissions/submission_7.csv',
    'Sub 7 - LR GridSearch Tuned'
)

In [ ]:
#Submission 8 - SVM GridSearch Tuned (Best Model)
print("\n--- Submission 8: SVM GridSearch Tuned (Best Model) ---")

s8 = make_submission(
    best_svm_tuned,
    X_test_full,
    test_ids,
    'Submissions/submission_8.csv',
    'Sub 8 - SVM GridSearch Tuned (Best)'
)

In [ ]:
#Submission 9 - Random Forest RandomizedSearch Tuned
print("\n--- Submission 9: RF RandomizedSearch Tuned ---")

s9 = make_submission(
    best_rf_tuned,
    X_test_full,
    test_ids,
    'Submissions/submission_9.csv',
    'Sub 9 - RF RandomizedSearch Tuned'
)

print("\n All 9 submission files generated")

## Save Best Model

In [ ]:
# Re-train best model on full training data
print(f"Re-training {best_name} on full dataset for web app...")
best_model.fit(X_train_full, y)

# Save all necessary artifacts
joblib.dump(best_model,  'Models/best_model.pkl')
joblib.dump(tfidf,       'Models/tfidf_vectorizer.pkl')
joblib.dump(NUMERIC_COLS,'Models/numeric_cols.pkl')
joblib.dump(best_name,   'Models/best_model_name.pkl')

print(f"Saved: best_model.pkl        ({best_name})")
print(f"Saved: tfidf_vectorizer.pkl  (5000 TF-IDF features)")
print(f"Saved: numeric_cols.pkl")
